In [1]:
!nvidia-smi
!nvidia-smi --query-gpu=name,compute_cap --format=csv
!nvcc --version

Mon Oct 26 14:48:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Exercise 01 — Timing a GPU Program
## Part (a): 512 blocks with one thread each

In [2]:
%%writefile ex1_timed.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>
#include <cuda_runtime.h>

#define N 512

__global__ void add(const int *a, const int *b, int *c)
{
    c[blockIdx.x] = a[blockIdx.x] + b[blockIdx.x];
}

void random_ints(int *x, int count)
{
    for (int i = 0; i < count; ++i)
        x[i] = rand() % 100;
}

double cpuSecond()
{
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

int main()
{
    size_t bytes = N * sizeof(int);

    int *a = (int *)malloc(bytes);
    int *b = (int *)malloc(bytes);
    int *c = (int *)malloc(bytes);

    if (!a || !b || !c) {
        fprintf(stderr, "Host allocation failed.\n");
        free(a);
        free(b);
        free(c);
        return EXIT_FAILURE;
    }

    random_ints(a, N);
    random_ints(b, N);

    int *d_a, *d_b, *d_c;
    cudaMalloc((void **)&d_a, bytes);
    cudaMalloc((void **)&d_b, bytes);
    cudaMalloc((void **)&d_c, bytes);

    cudaMemcpy(d_a, a, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, bytes, cudaMemcpyHostToDevice);

    double start = cpuSecond();

    add<<<N, 1>>>(d_a, d_b, d_c);
    cudaDeviceSynchronize();

    double elapsed = cpuSecond() - start;

    cudaMemcpy(c, d_c, bytes, cudaMemcpyDeviceToHost);

    for (int i = 0; i < 5; ++i)
        printf("%d + %d = %d\n", a[i], b[i], c[i]);

    int errors = 0;
    for (int i = 0; i < N; ++i) {
        if (c[i] != a[i] + b[i])
            ++errors;
    }

    printf("Verification: %d mismatches out of %d\n", errors, N);
    printf("Time taken - %.7f seconds\n", elapsed);

    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);

    free(a);
    free(b);
    free(c);

    return errors == 0 ? EXIT_SUCCESS : EXIT_FAILURE;
}

Writing ex1_timed.cu


In [3]:
!nvcc -arch=sm_75 ex1_timed.cu -o ex1_timed && ./ex1_timed > ex1_output.txt && cat ex1_output.txt

83 + 77 = 160
86 + 34 = 120
77 + 90 = 167
15 + 26 = 41
93 + 24 = 117
Verification: 0 mismatches out of 512
Time taken - 0.0005760 seconds


## Experiment — Timing without cudaDeviceSynchronize()

In [4]:
from pathlib import Path

code = Path("ex1_timed.cu").read_text()
code = code.replace(
    "    cudaDeviceSynchronize();",
    "    // Synchronization removed for this experiment."
)

Path("ex1_no_sync.cu").write_text(code)
print("Created ex1_no_sync.cu")

Created ex1_no_sync.cu


In [5]:
!nvcc -arch=sm_75 ex1_no_sync.cu -o ex1_no_sync && ./ex1_no_sync > ex1_no_sync_output.txt && cat ex1_no_sync_output.txt

83 + 77 = 160
86 + 34 = 120
77 + 90 = 167
15 + 26 = 41
93 + 24 = 117
Verification: 0 mismatches out of 512
Time taken - 0.0000939 seconds


## Part (b) — Timing Lab09 Exercises 3, 4, 5 and 6

In [6]:
from google.colab import files

uploaded = files.upload()

Saving SE3082_Lab09_IT24610781.ipynb to SE3082_Lab09_IT24610781.ipynb


In [7]:
import json
from pathlib import Path

notebooks = [name for name in uploaded if name.endswith(".ipynb")]
if len(notebooks) != 1:
    raise ValueError("Please upload exactly one Lab09 .ipynb notebook.")

notebook = json.loads(uploaded[notebooks[0]])

wanted = {"ex3_blocks.cu", "ex5_vecmul.cu", "ex6_matrix.cu"}
found = set()

for cell in notebook["cells"]:
    if cell["cell_type"] != "code":
        continue

    source = cell["source"]
    text = "".join(source) if isinstance(source, list) else source
    lines = text.splitlines()

    if not lines:
        continue

    command = lines[0].strip().split()
    if len(command) == 2 and command[0] == "%%writefile":
        name = command[1]
        if name in wanted:
            Path(name).write_text("\n".join(lines[1:]) + "\n")
            found.add(name)

missing = wanted - found
if missing:
    raise ValueError(f"Sources missing from notebook: {sorted(missing)}")

# Lab09 Exercise04 was generated from Exercise03.
code = Path("ex3_blocks.cu").read_text()
code = code.replace("int i = blockIdx.x;", "int i = threadIdx.x;")
code = code.replace("add<<<N, 1>>>", "add<<<1, N>>>")
code = code.replace(
    'printf("Launch: %d blocks x 1 thread = %d threads\\n", N, N);',
    'printf("Launch: 1 block x %d threads = %d threads\\n", N, N);'
)
Path("ex4_threads.cu").write_text(code)

for name in [
    "ex3_blocks.cu",
    "ex4_threads.cu",
    "ex5_vecmul.cu",
    "ex6_matrix.cu",
]:
    print(f"Ready: {name}")

Ready: ex3_blocks.cu
Ready: ex4_threads.cu
Ready: ex5_vecmul.cu
Ready: ex6_matrix.cu


Predictions:
1. The single block of 512 threads should use warp lanes more efficiently
   than 512 blocks containing one thread each. However, one block uses
   only one multiprocessor, and launch overhead may dominate this small task.

2. The 100-million-element matrix kernel may take roughly ten times
   as long as the 10-million-element vector kernel because both perform
   similar work per element. Actual scaling depends on memory bandwidth
   and GPU utilization.

In [8]:
from pathlib import Path

programs = {
    "ex3_blocks.cu": "add<<<N, 1>>>(d_a, d_b, d_c);",
    "ex4_threads.cu": "add<<<1, N>>>(d_a, d_b, d_c);",
    "ex5_vecmul.cu":
        "vecMul<<<blocks, THREADS_PER_BLOCK>>>(d_a, d_b, d_c, N);",
    "ex6_matrix.cu":
        "mulElem<<<numBlocks, threadsPerBlock>>>(d_a, d_b, d_c, ROWS, COLS);",
}

timer = """
double cpuSecond()
{
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

"""

for filename, launch in programs.items():
    code = Path(filename).read_text()

    # Normalize whitespace so the multiline matrix launch also matches.
    import re
    pattern = r"\s+".join(re.escape(part) for part in launch.split())
    matches = list(re.finditer(pattern, code))

    if len(matches) != 1:
        raise ValueError(f"Expected one kernel launch in {filename}")

    code = "#include <sys/time.h>\n" + code

    if "int main()" not in code:
        raise ValueError(f"Cannot locate main() in {filename}")

    code = code.replace("int main()", timer + "int main()", 1)

    timed_launch = f"""
    double start = cpuSecond();

    {launch}
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaDeviceSynchronize());

    double elapsed = cpuSecond() - start;
    printf("Kernel time = %.9f seconds\\n", elapsed);
"""

    code = re.sub(pattern, lambda match: timed_launch, code, count=1)

    output = filename.replace(".cu", "_timed.cu")
    Path(output).write_text(code)
    print(f"Created: {output}")

Created: ex3_blocks_timed.cu
Created: ex4_threads_timed.cu
Created: ex5_vecmul_timed.cu
Created: ex6_matrix_timed.cu


In [9]:
import csv
import re
import subprocess
from pathlib import Path

names = [
    "ex3_blocks",
    "ex4_threads",
    "ex5_vecmul",
    "ex6_matrix",
]

measurements = []

for name in names:
    executable = name + "_timed"

    subprocess.run(
        ["nvcc", "-arch=sm_75", executable + ".cu", "-o", executable],
        check=True
    )

    for run in range(1, 4):
        result = subprocess.run(
            ["./" + executable],
            capture_output=True,
            text=True
        )

        log = result.stdout + result.stderr
        Path(f"{executable}_run{run}.txt").write_text(log)

        if result.returncode != 0:
            raise RuntimeError(log)

        timing = re.search(r"Kernel time = ([0-9.]+) seconds", log)
        verification = re.search(
            r"Verification: (\d+) mismatches out of (\d+)", log
        )

        if not timing or not verification:
            raise RuntimeError(f"Missing timing or verification: {name}")

        if int(verification.group(1)) != 0:
            raise RuntimeError(f"Verification failed: {name}")

        seconds = float(timing.group(1))
        measurements.append([name, run, seconds])

        print(
            f"{name}, run {run}: {seconds:.9f} s | "
            f"0 mismatches out of {verification.group(2)}",
            flush=True
        )

with open("ex1b_timings.csv", "w", newline="") as stream:
    writer = csv.writer(stream)
    writer.writerow(["program", "run", "kernel_seconds"])
    writer.writerows(measurements)

print("\nSaved ex1b_timings.csv and all 12 output logs.")

ex3_blocks, run 1: 0.000097990 s | 0 mismatches out of 512
ex3_blocks, run 2: 0.000135899 s | 0 mismatches out of 512
ex3_blocks, run 3: 0.000087976 s | 0 mismatches out of 512
ex4_threads, run 1: 0.000092030 s | 0 mismatches out of 512
ex4_threads, run 2: 0.000101089 s | 0 mismatches out of 512
ex4_threads, run 3: 0.000089884 s | 0 mismatches out of 512
ex5_vecmul, run 1: 0.000560045 s | 0 mismatches out of 10000000
ex5_vecmul, run 2: 0.000561953 s | 0 mismatches out of 10000000
ex5_vecmul, run 3: 0.000579119 s | 0 mismatches out of 10000000
ex6_matrix, run 1: 0.004930019 s | 0 mismatches out of 100000000
ex6_matrix, run 2: 0.004945040 s | 0 mismatches out of 100000000
ex6_matrix, run 3: 0.004933834 s | 0 mismatches out of 100000000

Saved ex1b_timings.csv and all 12 output logs.


## Part (c) — Kernel Time versus Whole-sequence Time

In [10]:
from pathlib import Path

code = Path("ex5_vecmul_timed.cu").read_text()

changes = [
    (
        "#define N 10000000",
        "#define N 100000000"
    ),
    (
        "    CUDA_CHECK(cudaMemcpy(d_a, a, bytes, cudaMemcpyHostToDevice));",
        """    double totalStart = cpuSecond();

    CUDA_CHECK(cudaMemcpy(d_a, a, bytes, cudaMemcpyHostToDevice));"""
    ),
    (
        '    printf("Kernel time = %.9f seconds\\n", elapsed);',
        ""
    ),
    (
        "    CUDA_CHECK(cudaMemcpy(c, d_c, bytes, cudaMemcpyDeviceToHost));",
        """    CUDA_CHECK(cudaMemcpy(c, d_c, bytes, cudaMemcpyDeviceToHost));

    double totalElapsed = cpuSecond() - totalStart;

    printf("Kernel time = %.9f seconds\\n", elapsed);
    printf("Whole-sequence time = %.9f seconds\\n", totalElapsed);"""
    ),
]

for old, new in changes:
    if code.count(old) != 1:
        raise ValueError(f"Expected exactly one occurrence of: {old}")
    code = code.replace(old, new, 1)

# Keep printing outside the measured sequence.
old_print = '''    printf("Launching %d blocks x %d threads\\n",
           blocks, THREADS_PER_BLOCK);'''

if code.count(old_print) != 1:
    raise ValueError("Could not locate the launch information print.")

code = code.replace(old_print, "", 1)

Path("ex1_copy_timing.cu").write_text(code)
print("Created ex1_copy_timing.cu with 100,000,000 elements.")

Created ex1_copy_timing.cu with 100,000,000 elements.


In [11]:
!nvcc -arch=sm_75 ex1_copy_timing.cu -o ex1_copy_timing && ./ex1_copy_timing > ex1_copy_timing_output.txt && head -n 2 ex1_copy_timing_output.txt && tail -n 2 ex1_copy_timing_output.txt

Kernel time = 0.004692793 seconds
Whole-sequence time = 0.437891006 seconds
99999999) 86 x 21 = 1806
Verification: 0 mismatches out of 100000000


The whole copy-in, compute, copy-out sequence took approximately
93.3 times the kernel time. For this simple element-wise operation,
moving data dominates the measured sequence.

GPU work is more worthwhile when substantial computation is performed
per transferred element, or when data stays on the GPU for several
operations.

# Exercise 02 — Catching CUDA Errors with CHECK()

In [12]:
import re
from pathlib import Path

code = Path("ex1_timed.cu").read_text()

check_macro = r"""
#define CHECK(call) do {                                           \
    const cudaError_t error = (call);                              \
    if (error != cudaSuccess) {                                   \
        fprintf(stderr, "Error: %s:%d, code:%d, reason: %s\n",      \
                __FILE__, __LINE__, (int)error,                    \
                cudaGetErrorString(error));                       \
        exit(EXIT_FAILURE);                                       \
    }                                                             \
} while (0)

"""

code = code.replace("#define N 512", check_macro + "#define N 512", 1)

# Wrap the runtime calls in the original program.
pattern = r"(?m)^(\s*)(cuda(?:Malloc|Memcpy|DeviceSynchronize|Free)\([^\n]*\));"

code, wrapped = re.subn(
    pattern,
    lambda match: f"{match.group(1)}CHECK({match.group(2)});",
    code
)

if wrapped != 10:
    raise ValueError(f"Expected 10 CUDA calls; found {wrapped}")

launch = "add<<<N, 1>>>(d_a, d_b, d_c);"
if code.count(launch) != 1:
    raise ValueError("Could not locate the kernel launch.")

code = code.replace(
    launch,
    launch + "\n    CHECK(cudaGetLastError());",
    1
)

Path("ex2_check.cu").write_text(code)
print("Created ex2_check.cu with all CUDA calls checked.")

Created ex2_check.cu with all CUDA calls checked.


In [13]:
!nvcc -arch=sm_75 ex2_check.cu -o ex2_check && ./ex2_check > ex2_output.txt && cat ex2_output.txt

83 + 77 = 160
86 + 34 = 120
77 + 90 = 167
15 + 26 = 41
93 + 24 = 117
Verification: 0 mismatches out of 512
Time taken - 0.0001171 seconds


## Experiments A, B and C — Deliberate CUDA Errors

In [14]:
from pathlib import Path

checked = Path("ex2_check.cu").read_text()
plain = Path("ex1_timed.cu").read_text()

launch = "add<<<N, 1>>>(d_a, d_b, d_c);"
invalid_launch = "add<<<N, 2048>>>(d_a, d_b, d_c);"

# A: Exceed the threads-per-block limit, with error checking.
assert checked.count(launch) == 1
Path("ex2_a_threads.cu").write_text(
    checked.replace(launch, invalid_launch, 1)
)

# B: Request an impossibly large device allocation.
allocation = "CHECK(cudaMalloc((void **)&d_a, bytes));"
assert checked.count(allocation) == 1
Path("ex2_b_memory.cu").write_text(
    checked.replace(
        allocation,
        "CHECK(cudaMalloc((void **)&d_a, (size_t)1 << 50));",
        1
    )
)

# C: Repeat the invalid launch without CUDA error checking.
assert plain.count(launch) == 1
Path("ex2_c_no_check.cu").write_text(
    plain.replace(launch, invalid_launch, 1)
)

print("Created all three experiment files.")

Created all three experiment files.


In [15]:
import subprocess
from pathlib import Path

for name in ["ex2_a_threads", "ex2_b_memory", "ex2_c_no_check"]:
    subprocess.run(
        ["nvcc", "-arch=sm_75", name + ".cu", "-o", name],
        check=True
    )

    result = subprocess.run(
        ["./" + name],
        capture_output=True,
        text=True,
        timeout=30
    )

    log = (
        result.stdout
        + result.stderr
        + f"\nProgram exit code: {result.returncode}\n"
    )

    Path(name + "_output.txt").write_text(log)

    print(f"\n--- {name} ---")
    print(log)


--- ex2_a_threads ---
Error: ex2_a_threads.cu:67, code:1, reason: invalid argument

Program exit code: 1


--- ex2_b_memory ---
Error: ex2_b_memory.cu:57, code:2, reason: out of memory

Program exit code: 1


--- ex2_c_no_check ---
83 + 77 = 0
86 + 34 = 0
77 + 90 = 0
15 + 26 = 0
93 + 24 = 0
Verification: 512 mismatches out of 512
Time taken - 0.0001118 seconds

Program exit code: 1



# Exercise 03 — Scaling a Picture with a 2D Grid

Prediction: Original values from 128 through 255 exceed 255 when
doubled. These are half the possible input values, so approximately
240,000 of the 480,000 pixels should be capped.

In [16]:
%%writefile ex3_picture.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>
#include <cuda_runtime.h>

#define WIDTH 800
#define HEIGHT 600

#define CHECK(call) do {                                           \
    const cudaError_t error = (call);                              \
    if (error != cudaSuccess) {                                   \
        fprintf(stderr, "Error: %s:%d, code:%d, reason: %s\n",      \
                __FILE__, __LINE__, (int)error,                    \
                cudaGetErrorString(error));                       \
        exit(EXIT_FAILURE);                                       \
    }                                                             \
} while (0)

double cpuSecond()
{
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

__global__ void PictureKernel(const unsigned char *input,
                              unsigned char *output,
                              int height, int width)
{
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < height && col < width) {
        int idx = row * width + col;
        int value = 2 * input[idx];
        output[idx] = (value > 255) ? 255 : value;
    }
}

int main()
{
    size_t count = (size_t)WIDTH * HEIGHT;
    size_t bytes = count * sizeof(unsigned char);

    unsigned char *input = (unsigned char *)malloc(bytes);
    unsigned char *output = (unsigned char *)malloc(bytes);

    if (!input || !output) {
        fprintf(stderr, "Host allocation failed.\n");
        free(input);
        free(output);
        return EXIT_FAILURE;
    }

    for (size_t i = 0; i < count; ++i)
        input[i] = rand() % 256;

    unsigned char *d_input, *d_output;
    CHECK(cudaMalloc((void **)&d_input, bytes));
    CHECK(cudaMalloc((void **)&d_output, bytes));

    CHECK(cudaMemcpy(d_input, input, bytes,
                     cudaMemcpyHostToDevice));

    dim3 threadsPerBlock(16, 16);
    dim3 numBlocks(
        (WIDTH + threadsPerBlock.x - 1) / threadsPerBlock.x,
        (HEIGHT + threadsPerBlock.y - 1) / threadsPerBlock.y
    );

    printf("Grid: %u x %u blocks of %u x %u threads\n",
           numBlocks.x, numBlocks.y,
           threadsPerBlock.x, threadsPerBlock.y);

    double start = cpuSecond();

    PictureKernel<<<numBlocks, threadsPerBlock>>>(
        d_input, d_output, HEIGHT, WIDTH
    );
    CHECK(cudaGetLastError());
    CHECK(cudaDeviceSynchronize());

    double elapsed = cpuSecond() - start;

    CHECK(cudaMemcpy(output, d_output, bytes,
                     cudaMemcpyDeviceToHost));

    printf("Original top-left 50 x 50:\n");
    for (int row = 0; row < 50; ++row) {
        for (int col = 0; col < 50; ++col)
            printf("%3d ", (int)input[row * WIDTH + col]);
        printf("\n");
    }

    printf("Updated top-left 50 x 50:\n");
    for (int row = 0; row < 50; ++row) {
        for (int col = 0; col < 50; ++col)
            printf("%3d ", (int)output[row * WIDTH + col]);
        printf("\n");
    }

    size_t errors = 0;
    size_t capped = 0;

    for (size_t i = 0; i < count; ++i) {
        int expected = 2 * input[i];

        if (expected > 255) {
            expected = 255;
            ++capped;
        }

        if (output[i] != expected)
            ++errors;
    }

    printf("Verification: %zu mismatches out of %zu\n",
           errors, count);
    printf("Pixels capped at 255: %zu\n", capped);
    printf("Time taken - %.7f seconds\n", elapsed);

    CHECK(cudaFree(d_input));
    CHECK(cudaFree(d_output));
    free(input);
    free(output);

    return errors == 0 ? EXIT_SUCCESS : EXIT_FAILURE;
}

Writing ex3_picture.cu


In [17]:
!nvcc -arch=sm_75 ex3_picture.cu -o ex3_picture && ./ex3_picture > ex3_picture_output.txt && cat ex3_picture_output.txt

Grid: 50 x 38 blocks of 16 x 16 threads
Original top-left 50 x 50:
103 198 105 115  81 255  74 236  41 205 186 171 242 251 227  70 124 194  84 248  27 232 231 141 118  90  46  99  51 159 201 154 102  50  13 183  49  88 163  90  37  93   5  23  88 233  94 212 171 178 
 96  34  10 167  77  49 181  91   3 160  13  34  13  71  93 205 155 135 120  86 213 112  76 156 134 234  15 152 242 235 156  83  13 167 250  90 216 176 181 219  80 194 253  93   9  90  42 165 226 163 
210 229 220  71  16 152 132 214 161  59  36  81  31  29 107 245  90 125  16 216  23 252 165  61 140  36 239 252 218 206  78 172 179  42 243 196 195 119 154 100 178 190 181 209 219  32 198  53 157 214 
158 212 187 189 162 172 238 192 141 116  54 141  24 225  34 225 154   4  34 178 109 178 216 130 145 231 176 222 132 115 155  34  71  86 223 233   2 205 169 143  65 224  28  90 193  63  59  91  67  93 
219 240 232   2 175 227 102  92  27 167 209 145 153  42 245 250 103 153 124 186 196 109  26  59 117 143  79  87 135 187  33  98 1

# Exercise 04 — Element-wise Multiplication with a 3D Grid

In [18]:
%%writefile ex4_volume.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>
#include <cuda_runtime.h>

#define NX 500
#define NY 500
#define NZ 500

#define CHECK(call) do {                                           \
    const cudaError_t error = (call);                              \
    if (error != cudaSuccess) {                                   \
        fprintf(stderr, "Error: %s:%d, code:%d, reason: %s\n",      \
                __FILE__, __LINE__, (int)error,                    \
                cudaGetErrorString(error));                       \
        exit(EXIT_FAILURE);                                       \
    }                                                             \
} while (0)

double cpuSecond()
{
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

__global__ void mulElem3D(const int *a, const int *b, int *c,
                          int nx, int ny, int nz)
{
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;
    int z = blockIdx.z * blockDim.z + threadIdx.z;

    if (x < nx && y < ny && z < nz) {
        size_t idx = ((size_t)z * ny + y) * nx + x;
        c[idx] = a[idx] * b[idx];
    }
}

void random_ints(int *array, size_t count)
{
    for (size_t i = 0; i < count; ++i)
        array[i] = rand() % 100;
}

int main()
{
    size_t count = (size_t)NX * NY * NZ;
    size_t bytes = count * sizeof(int);

    int *a = (int *)malloc(bytes);
    int *b = (int *)malloc(bytes);
    int *c = (int *)malloc(bytes);

    if (!a || !b || !c) {
        fprintf(stderr, "Host allocation failed.\n");
        free(a);
        free(b);
        free(c);
        return EXIT_FAILURE;
    }

    random_ints(a, count);
    random_ints(b, count);

    int *d_a, *d_b, *d_c;

    CHECK(cudaMalloc((void **)&d_a, bytes));
    CHECK(cudaMalloc((void **)&d_b, bytes));
    CHECK(cudaMalloc((void **)&d_c, bytes));

    CHECK(cudaMemcpy(d_a, a, bytes, cudaMemcpyHostToDevice));
    CHECK(cudaMemcpy(d_b, b, bytes, cudaMemcpyHostToDevice));

    dim3 threadsPerBlock(8, 8, 8);
    dim3 numBlocks(
        (NX + threadsPerBlock.x - 1) / threadsPerBlock.x,
        (NY + threadsPerBlock.y - 1) / threadsPerBlock.y,
        (NZ + threadsPerBlock.z - 1) / threadsPerBlock.z
    );

    printf("Grid: %u x %u x %u blocks\n",
           numBlocks.x, numBlocks.y, numBlocks.z);
    printf("Threads per block: %u x %u x %u = %u\n",
           threadsPerBlock.x, threadsPerBlock.y,
           threadsPerBlock.z,
           threadsPerBlock.x * threadsPerBlock.y * threadsPerBlock.z);

    double start = cpuSecond();

    mulElem3D<<<numBlocks, threadsPerBlock>>>(
        d_a, d_b, d_c, NX, NY, NZ
    );
    CHECK(cudaGetLastError());
    CHECK(cudaDeviceSynchronize());

    double elapsed = cpuSecond() - start;

    CHECK(cudaMemcpy(c, d_c, bytes, cudaMemcpyDeviceToHost));

    for (size_t idx = count - 1000; idx < count; ++idx) {
        size_t x = idx % NX;
        size_t y = (idx / NX) % NY;
        size_t z = idx / ((size_t)NX * NY);

        printf("C[%zu][%zu][%zu] = %d x %d = %d\n",
               z, y, x, a[idx], b[idx], c[idx]);
    }

    size_t errors = 0;
    for (size_t i = 0; i < count; ++i) {
        if (c[i] != a[i] * b[i])
            ++errors;
    }

    printf("Verification: %zu mismatches out of %zu\n",
           errors, count);
    printf("Time taken - %.7f seconds\n", elapsed);

    CHECK(cudaFree(d_a));
    CHECK(cudaFree(d_b));
    CHECK(cudaFree(d_c));

    free(a);
    free(b);
    free(c);

    return errors == 0 ? EXIT_SUCCESS : EXIT_FAILURE;
}

Writing ex4_volume.cu


In [19]:
!nvcc -arch=sm_75 ex4_volume.cu -o ex4_volume && ./ex4_volume > ex4_volume_output.txt && head -n 3 ex4_volume_output.txt && tail -n 3 ex4_volume_output.txt

Grid: 63 x 63 x 63 blocks
Threads per block: 8 x 8 x 8 = 512
C[499][498][0] = 76 x 15 = 1140
C[499][499][499] = 47 x 68 = 3196
Verification: 0 mismatches out of 125000000
Time taken - 0.0075321 seconds
